## **Important Information:**
1.   The pretraied model is loaded in the variable `model`.
2.   All clean images and labels are loaded as tensors `X` and `y_true` respectively such that `y_true[i]` is the actual label of `X[i]`.
3.   To predict all labels call `model(X)`.
4.   Please properly follow the instructions corresponding to each of the following cells.

## **Specify the data path**
In the following cell provide appropriate path to the "p2_data.zip" file.

In [ ]:
# EDIT THE PATH IF NEEDED
zip_path = "p2_data.zip" # path to the "p2_data.zip" file.

## **Load data file**
**Do not edit the following cell.**

In [ ]:
# DO NOT EDIT THIS CELL

import zipfile

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall()

with open("./p2_data/readme.txt", "r") as f:
    print(f.read())

## **Prepare the model and data**
1. **Do not edit the following cell.**
2. The variable `class_names` contains name of the output labels.
3. The variables `mean` and `std` are used to normalize the input images using mean and standard deviation.



In [ ]:
# DO NOT EDIT THIS CELL

import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

mean = (0.4914, 0.4822, 0.4465) # image mean
std  = (0.2470, 0.2435, 0.2616) # image standard deviation
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
img_per_class = 100

# prepare input images
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)]) # specify transformations: covert to tensor and normalize
paths = [f"./p2_data/clean_images/{name}_{i}.png" for name in class_names for i in range(img_per_class)]
labels = [class_names.index(name) for name in class_names for _ in range(img_per_class)]
X = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths], dim = 0).to(device) # convert to tensor and normalize
y_true = torch.tensor(labels).to(device) # convert to tensor

# load cnn model
model = torch.jit.load("./p2_data/cnn_model.pt", map_location = device)

# sanity check
y_pred = model(X)
matches = (y_true == y_pred)
print(f"Total images: {len(y_true)}  |  Correctly Classified: {sum(matches).item()}")

# show some images
print("Showing some sample images ...")
mean_t = torch.tensor(mean).view(3, 1, 1)
std_t  = torch.tensor(std).view(3, 1, 1)
fig, axes = plt.subplots(2, 5, figsize=(9, 3.5))
for i, idx in enumerate([59, 119, 283, 360, 427, 527, 682, 783, 806, 991]):
    img = (X[idx].cpu() * std_t + mean_t).clamp(0, 1)
    img = img.permute(1, 2, 0).numpy()
    ax = axes[i // 5, i % 5]
    ax.imshow(img)
    ax.set_title(f"{class_names[idx // img_per_class]} (label: {labels[idx]})", fontsize = 10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## **Adversarial Attack**
Implement your attack in the following cell.
1. You can only call `model()` as a **black box oracle** and use the output.
2. Your method **must not** see the actual parameters or architecture.
3. Your method **must not** see the intermediate layer outputs including class probabilities.

In [ ]:
# =====================================================================
#  BLACK-BOX (HARD-LABEL) ADVERSARIAL ATTACK
# =====================================================================
#  Threat model actually used by the code below:
#    * the only thing read from `model` is the integer label it returns;
#    * no architecture, no parameters, no logits, no intermediate outputs.
#  Every oracle call goes through `Oracle.predict`, which counts queries,
#  so the inference budget reported at the end is exact.
#
#  Pipeline
#  --------
#   0   free adversarial seeds: clean images of *other* classes (0 queries).
#       All 1000 provided images are classified correctly, so a clean image
#       with a different label is already adversarial for this one -- and it
#       is a valid 8-bit PNG, which puts a 100% success floor under the run.
#   1   bisection onto the decision boundary.
#   1b  substitute networks, trained on labels *queried from the victim*,
#       supply candidate directions; the victim's own labels then decide the
#       smallest radius along each (Papernot et al. 2017; Rony et al. 2019).
#       Set CONFIG["transfer"] = False for the pure query-only variant.
#   2   HopSkipJump: estimate the boundary normal from +-1 labels, step,
#       re-project (Chen, Jordan & Wainwright, IEEE S&P 2020).  Skipped when
#       CONFIG["iters"] == 0, which the transfer stage makes possible.
#   3   boundary refinement: geometric steps (SurFree, Maho et al. 2021) and
#       a random walk along the boundary (Brendel et al. 2018), then extra
#       passes spent only on the images still furthest away.
#   4   8-bit finalisation: the perturbation is about one pixel level in RMS,
#       so it has to be made to survive rounding to the PNG grid.
#   4b  refinement of the *rounded* objective directly -- the quantity that
#       actually gets graded.
#   5   integer-grid polishing: L-inf clipping, pruning, greedy shrinking.
#   6   verification across several evaluators (different batch sizes, CPU as
#       well as GPU) and both memory layouts, including the exact
#       representation a PNG loader hands back.
# =====================================================================

import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

# the pixel-space normalisation constants, under the names the inlined
# substitute-model code expects
MEAN, STD = mean, std

CONFIG = {
    'seeds': 4,
    'bsearch': 20,
    'transfer': True,
    'surrogate_epochs': 20,
    'iters': 0,
    'b0': 100,
    'bmax': 400,
    'refine_rounds': 1,
    'geo_steps': 2000,
    'walk_steps': 2000,
    'rescue_rounds': 2,
    'rescue_frac': 0.3,
    'rescue_geo': 700,
    'rescue_walk': 700,
    'grid_rounds': 1,
    'grid_geo': 800,
    'grid_walk': 800,
    'greedy_rounds': 5,
    'max_batch': 2048,
    'verify_shrink': 0.97,
    'seed': 0,
    'model_path': './p2_data/cnn_model.pt',
}

class Oracle:
    """Hard-label-only, query-counting wrapper around the provided model.

    Inputs are images in pixel space: float tensors in [0, 1] of shape
    (N, 3, 32, 32).  Normalisation with (mean, std) happens inside, exactly as
    in the `transform` used to build `X`.
    """

    def __init__(self, model, mean, std, device="cpu", batch_size=2048,
                 quantize=False):
        self.model, self.device, self.batch_size = model, device, batch_size
        self.quantize = quantize
        self.n_queries = 0
        self._mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
        self._std = torch.tensor(std, device=device).view(1, 3, 1, 1)

    @staticmethod
    def quantize_pixels(x01):
        """Snap to the 8-bit grid a .png file can hold."""
        return torch.round(x01.clamp(0.0, 1.0) * 255.0) / 255.0

    def reset_counter(self):
        self.n_queries = 0

    @torch.no_grad()
    def predict(self, x01):
        x01 = x01.to(self.device)
        x01 = self.quantize_pixels(x01) if self.quantize else x01.clamp(0.0, 1.0)
        out = []
        for i in range(0, x01.shape[0], self.batch_size):
            chunk = x01[i:i + self.batch_size]
            lab = self.model((chunk - self._mean) / self._std)
            if lab.dim() > 1:          # defensive: use the arg-max only,
                lab = lab.argmax(dim=1)  # never the scores themselves
            out.append(lab.reshape(-1))
            self.n_queries += chunk.shape[0]
        return torch.cat(out).to(torch.long)

    @torch.no_grad()
    def is_adversarial(self, x01, y_true):
        return self.predict(x01) != y_true.to(self.device)





# --------------------------------------------------------------------- helpers
def l2(v: torch.Tensor) -> torch.Tensor:
    """Per-image L2 norm of a (N,C,H,W) tensor -> (N,)."""
    return v.flatten(1).norm(dim=1)


def linf(v: torch.Tensor) -> torch.Tensor:
    return v.flatten(1).abs().max(dim=1).values


def _b(v: torch.Tensor) -> torch.Tensor:
    """(N,) -> (N,1,1,1) for broadcasting against images."""
    return v.view(-1, 1, 1, 1)


class BestTracker:
    """Keeps, per image, the closest point that the oracle *verified* as
    adversarial.  Guarantees monotone improvement and a 100% success rate as
    long as it is seeded with a verified adversarial point."""

    def __init__(self, x_orig: torch.Tensor, quantize: bool = False):
        self.x_orig = x_orig
        self.best = x_orig.clone()
        self.dist = torch.full((x_orig.shape[0],), float("inf"), device=x_orig.device)
        # When True, candidates are snapped to the 8-bit grid before being
        # measured and stored, so the tracked distance is the distance of the
        # image that will actually be written to the .png.  Pair this with an
        # Oracle constructed with quantize=True, or the stored point and the
        # point the oracle answered about are not the same image.
        self.quantize = quantize

    def update(self, x_cand: torch.Tensor, is_adv: torch.Tensor,
               idx: torch.Tensor | None = None) -> torch.Tensor:
        """`x_cand`/`is_adv` are aligned with `idx` (default: all images)."""
        if idx is None:
            idx = torch.arange(x_cand.shape[0], device=x_cand.device)
        if self.quantize:
            x_cand = torch.round(x_cand.clamp(0, 1) * 255.0) / 255.0
        d = l2(x_cand - self.x_orig[idx])
        take = is_adv & (d < self.dist[idx])
        sel = idx[take]
        if sel.numel():
            self.best[sel] = x_cand[take]
            self.dist[sel] = d[take]
        return take


# ------------------------------------------------------------ initialisation
def init_seeds(x01: torch.Tensor, y: torch.Tensor, n_seeds: int = 4):
    """Zero-query adversarial starting points.

    Every clean image is classified correctly by the victim model (stated in
    the provided notebook and re-verified in `run_attack.py`), so *any* clean
    image whose label differs from `y[i]` is, by construction, an adversarial
    example for image `i`.  Picking the nearest such images in pixel space
    gives a strong starting point for free -- no queries are spent on the
    usual "sample random noise until it is misclassified" phase.

    Returns a (N, n_seeds, C, H, W) tensor of candidate seeds.
    """
    n = x01.shape[0]
    flat = x01.flatten(1)
    seeds = torch.empty(n, n_seeds, *x01.shape[1:], device=x01.device)
    # chunk the distance matrix to stay inside memory
    step = 128
    for s in range(0, n, step):
        e = min(n, s + step)
        d = torch.cdist(flat[s:e], flat)                      # (m, n)
        same = y[s:e].view(-1, 1) == y.view(1, -1)
        d = d.masked_fill(same, float("inf"))
        idx = d.topk(n_seeds, dim=1, largest=False).indices   # (m, n_seeds)
        seeds[s:e] = x01[idx]
    return seeds


def best_initial_boundary_point(oracle, x01, y, seeds, bsearch_steps=10,
                               tracker: BestTracker | None = None):
    """Binary-search every seed onto the boundary, keep the closest one."""
    n, k = seeds.shape[0], seeds.shape[1]
    best = None
    best_d = torch.full((n,), float("inf"), device=x01.device)
    for j in range(k):
        pt, _ = line_search_boundary(oracle, x01, seeds[:, j], y, bsearch_steps,
                                     tracker=tracker)
        d = l2(pt - x01)
        if best is None:
            best, best_d = pt, d
        else:
            upd = d < best_d
            best[upd] = pt[upd]
            best_d[upd] = d[upd]
    return best, best_d


# ------------------------------------------------------------- boundary search
def line_search_boundary(oracle, x_orig, x_adv, y, steps=10,
                         tracker: BestTracker | None = None):
    """Smallest alpha in (0,1] with `x_orig + alpha*(x_adv-x_orig)` adversarial.

    `x_adv` must be adversarial (alpha = 1 is the known-good bracket).
    Returns (point_on_boundary, alpha).
    """
    n = x_orig.shape[0]
    lo = torch.zeros(n, device=x_orig.device)
    hi = torch.ones(n, device=x_orig.device)
    diff = x_adv - x_orig
    for _ in range(steps):
        mid = 0.5 * (lo + hi)
        z = (x_orig + _b(mid) * diff).clamp(0, 1)
        adv = oracle.is_adversarial(z, y)
        if tracker is not None:
            tracker.update(z, adv)
        hi = torch.where(adv, mid, hi)
        lo = torch.where(adv, lo, mid)
    return (x_orig + _b(hi) * diff).clamp(0, 1), hi


# -------------------------------------------------- label-only normal estimate
def sample_directions(m, n_samples, shape, device, k=0, generator=None):
    """Unit-norm random probe directions.

    `k == 0` gives i.i.d. Gaussian directions in the full 3*32*32 space
    (plain HopSkipJump).  `k > 0` draws them in a (3*k*k)-dimensional
    *smooth* subspace -- low-resolution noise bilinearly upsampled to full
    resolution -- which is the query-efficiency trick of QEBA (Li et al.,
    CVPR 2020).  Estimating a direction from +-1 labels needs O(dim) samples,
    so shrinking the dimension from 3072 to e.g. 3*8*8 = 192 buys a far
    better direction per query.
    """
    c, h, w = shape
    if k and k < min(h, w):
        v = torch.randn(m * n_samples, c, k, k, device=device, generator=generator)
        u = torch.nn.functional.interpolate(v, size=(h, w), mode="bilinear",
                                            align_corners=False)
        u = u.view(m, n_samples, c, h, w)
    else:
        u = torch.randn(m, n_samples, c, h, w, device=device, generator=generator)
    return u / (u.flatten(2).norm(dim=2).view(m, n_samples, 1, 1, 1) + 1e-12)


def estimate_gradient(oracle, x_adv, y, delta, n_samples, max_batch=16384,
                      subspace_k=0, generator=None, sampler=None):
    """Monte-Carlo estimate of the (outward) boundary normal at `x_adv`.

    Uses only the +-1 "is it adversarial" answers, as in HopSkipJump.
    """
    n = x_adv.shape[0]
    shape = x_adv.shape[1:]
    grads = torch.empty_like(x_adv)
    group = max(1, max_batch // n_samples)
    for s in range(0, n, group):
        e = min(n, s + group)
        m = e - s
        xa = x_adv[s:e]
        dl = delta[s:e].view(m, 1, 1, 1, 1)
        if sampler is None:
            u = sample_directions(m, n_samples, shape, x_adv.device,
                                  k=subspace_k, generator=generator)
        else:
            u = sampler(m, n_samples, shape, x_adv.device, xa, y[s:e],
                        k=subspace_k, generator=generator)
        pert = (xa.unsqueeze(1) + dl * u).clamp(0, 1)
        # account for the clipping, as in the reference implementation
        u = (pert - xa.unsqueeze(1)) / dl
        yy = y[s:e].repeat_interleave(n_samples)
        adv = oracle.is_adversarial(pert.reshape(m * n_samples, *shape), yy)
        f = adv.view(m, n_samples).float() * 2.0 - 1.0
        fbar = f.mean(dim=1, keepdim=True)
        w = f - fbar
        degenerate = fbar.abs() >= 1.0 - 1e-6        # all-adv or all-clean
        w = torch.where(degenerate, fbar.sign(), w)
        grads[s:e] = (w.view(m, n_samples, 1, 1, 1) * u).mean(dim=1)
    return grads / (_b(l2(grads)) + 1e-12)


def geometric_step(oracle, x_adv, grad, y, eps, max_halvings=12,
                   tracker: BestTracker | None = None):
    """Move along `grad` with a geometrically decreasing step until the point
    is adversarial again.  Only the still-unsuccessful images are queried."""
    n = x_adv.shape[0]
    eps = eps.clone()
    out = x_adv.clone()
    pending = torch.ones(n, dtype=torch.bool, device=x_adv.device)
    for _ in range(max_halvings):
        idx = pending.nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            break
        cand = (x_adv[idx] + _b(eps[idx]) * grad[idx]).clamp(0, 1)
        ok = oracle.is_adversarial(cand, y[idx])
        if tracker is not None:
            tracker.update(cand, ok, idx)
        out[idx[ok]] = cand[ok]
        pending[idx[ok]] = False
        eps[idx[~ok]] *= 0.5
    return out, ~pending


# ------------------------------------------------------------------ HopSkipJump
def hsja(oracle, x_orig, y, x_adv, iters=30, b0=100, b_max=1024,
         bsearch_steps=20, gamma=1.0, tracker: BestTracker | None = None,
         max_batch=16384, subspace_schedule=None, sampler=None, log=None):
    """HopSkipJump refinement of an adversarial point, L2 objective.

    `bsearch_steps` must be fine enough that the re-projected point sits much
    closer to the boundary than the probe radius `delta`; otherwise the +-1
    answers saturate and the estimated normal is useless.  For d = 3072 the
    HopSkipJump tolerance theta = 1/d^1.5 needs ~18 bisection steps.

    `subspace_schedule` is a list of (last_iteration, k) pairs; `k = 0` means
    the full pixel space.  Coarse subspaces make fast progress early, the full
    space refines the last bit of detail.
    """
    d = x_orig[0].numel()
    theta = gamma / (d ** 1.5)
    if tracker is None:
        tracker = BestTracker(x_orig)
    if subspace_schedule is None:
        subspace_schedule = [(iters, 0)]

    def k_for(t):
        for last, k in subspace_schedule:
            if t <= last:
                return k
        return subspace_schedule[-1][1]

    for t in range(1, iters + 1):
        dist = l2(x_adv - x_orig)
        delta = (math.sqrt(d) * theta * dist).clamp(min=1e-7)
        n_samples = int(min(b0 * math.sqrt(t), b_max))
        k = k_for(t)

        grad = estimate_gradient(oracle, x_adv, y, delta, n_samples,
                                 max_batch=max_batch, subspace_k=k,
                                 sampler=sampler)
        eps = dist / math.sqrt(t)
        moved, _ = geometric_step(oracle, x_adv, grad, y, eps, tracker=tracker)
        x_adv, _ = line_search_boundary(oracle, x_orig, moved, y, bsearch_steps,
                                        tracker=tracker)
        tracker.update(x_adv, oracle.is_adversarial(x_adv, y))

        if log is not None:
            log(t, n_samples, k, tracker.dist, oracle.n_queries)

    return tracker


# ------------------------------------------------- boundary walk (fine polish)
def boundary_walk(oracle, x_orig, y, x_adv, steps=400, tracker=None,
                  subspace_k=0, sph_step=0.12, src_step=0.03,
                  sph_bounds=(0.005, 0.6), src_bounds=(1e-4, 0.4),
                  adapt=1.25, generator=None, log_every=0):
    """Random-walk refinement along the decision boundary (Brendel, Rauber &
    Bethge, ICLR 2018), vectorised over the image set.

    Near convergence this is far cheaper per unit of progress than another
    HopSkipJump iteration: one *orthogonal* trial plus one *inward* trial per
    image per step, both answered in a single batched oracle call, instead of
    several hundred probes for a new normal estimate.

    Per-image step sizes adapt to the observed acceptance.
    """
    if tracker is None:
        tracker = BestTracker(x_orig)
    n = x_orig.shape[0]
    dev = x_orig.device
    sph = torch.full((n,), sph_step, device=dev)
    src = torch.full((n,), src_step, device=dev)
    n_lat = n_in = 0

    for i in range(steps):
        diff = x_adv - x_orig
        dist = l2(diff).clamp(min=1e-9)

        # --- orthogonal (same-distance) trial --------------------------------
        u = sample_directions(n, 1, x_orig.shape[1:], dev, k=subspace_k,
                              generator=generator).squeeze(1)
        u = u * _b(dist * sph)
        # remove the radial component, then rescale back onto the sphere
        radial = (u * diff).flatten(1).sum(1) / dist ** 2
        u = u - _b(radial) * diff
        cand = x_orig + (diff + u)
        cand = x_orig + (cand - x_orig) * _b(dist / l2(cand - x_orig).clamp(min=1e-9))
        cand = cand.clamp(0, 1)

        # --- inward trial (does not depend on the outcome above, so both
        #     trials go to the oracle in one batch) --------------------------
        cand_in = (cand + _b(src) * (x_orig - cand)).clamp(0, 1)

        adv = oracle.is_adversarial(torch.cat([cand, cand_in], 0),
                                    torch.cat([y, y], 0))
        ok_lat, ok_in = adv[:n], adv[n:]
        tracker.update(cand, ok_lat)
        tracker.update(cand_in, ok_in)

        took_in = ok_lat & ok_in
        took_lat = ok_lat & ~ok_in
        x_adv = torch.where(_b(took_in), cand_in,
                            torch.where(_b(took_lat), cand, x_adv))

        # --- adapt ----------------------------------------------------------
        src = torch.where(took_in, src * adapt, src / adapt).clamp(*src_bounds)
        sph = torch.where(ok_lat, sph * 1.05, sph / 1.05).clamp(*sph_bounds)
        n_in += int(took_in.sum()); n_lat += int(took_lat.sum())

        if log_every and (i + 1) % log_every == 0:
            print(f"    [walk] {i+1:4d}/{steps}  L2med "
                  f"{(l2(x_adv-x_orig)*255).median():7.2f}  "
                  f"inward-acc {n_in/(n*(i+1)):.2f}  lateral-acc "
                  f"{n_lat/(n*(i+1)):.2f}  q={oracle.n_queries:,}")
    return tracker, x_adv


# --------------------------------------------- geometric (circular) refinement
def geometric_walk(oracle, x_orig, y, x_adv, steps=400, tracker=None,
                   thetas=(0.55, 0.3, 0.15), subspace_k=0, adapt=1.1,
                   theta_bounds=(0.02, 1.2), project_every=50, project_steps=7,
                   generator=None, log_every=0):
    """SurFree-style refinement (Maho, Furon & Le Merrer, CVPR 2021).

    With `u` the unit vector from the clean image to the current boundary
    point and `v` a random unit vector orthogonal to it,

        z(t) = x + d*cos(t) * ( cos(t)*u + sin(t)*v )

    satisfies ||z(t) - x|| = d*cos(t): *one* accepted query shrinks the
    distance by the factor cos(t).  Where the boundary curves away from the
    clean image this is far more productive per query than a small random
    walk, so several `t` values are tried per step in a single batch and the
    largest accepted one (the biggest shrink) is kept.
    """
    if tracker is None:
        tracker = BestTracker(x_orig)
    n = x_orig.shape[0]
    dev = x_orig.device
    k = len(thetas)
    scale = torch.ones(n, device=dev)        # per-image multiplier on `thetas`
    n_acc = 0

    for i in range(steps):
        diff = x_adv - x_orig
        dist = l2(diff).clamp(min=1e-9)
        u = diff / _b(dist)

        v = sample_directions(n, 1, x_orig.shape[1:], dev, k=subspace_k,
                              generator=generator).squeeze(1)
        v = v - _b((v * u).flatten(1).sum(1)) * u
        v = v / _b(l2(v).clamp(min=1e-12))

        cands, dists = [], []
        for t0 in thetas:
            t = (scale * t0).clamp(*theta_bounds)
            ct, st = torch.cos(t), torch.sin(t)
            z = x_orig + _b(dist * ct) * (_b(ct) * u + _b(st) * v)
            cands.append(z.clamp(0, 1))
            dists.append(dist * ct)
        batch = torch.cat(cands, 0)
        adv = oracle.is_adversarial(batch, y.repeat(k)).view(k, n)
        for j in range(k):
            tracker.update(cands[j], adv[j])

        # keep the largest accepted theta == the smallest resulting distance
        taken = torch.zeros(n, dtype=torch.bool, device=dev)
        for j in range(k):                   # thetas are in decreasing order
            pick = adv[j] & ~taken
            x_adv = torch.where(_b(pick), cands[j], x_adv)
            taken |= pick
        n_acc += int(taken.sum())

        # grow the search angle when it works, shrink it when nothing does
        scale = torch.where(adv[0], scale * adapt, scale)
        scale = torch.where(~taken, scale / adapt, scale).clamp(0.05, 20.0)

        # an accepted step lands *inside* the adversarial region, so every so
        # often bisect back along the radius to the smallest radius that still
        # works -- a few queries for a guaranteed reduction
        if project_every and (i + 1) % project_every == 0:
            x_adv, _ = line_search_boundary(oracle, x_orig, x_adv, y,
                                            project_steps, tracker=tracker)

        if log_every and (i + 1) % log_every == 0:
            print(f"    [geo ] {i+1:4d}/{steps}  L2med "
                  f"{(l2(x_adv-x_orig)*255).median():7.2f}  acc "
                  f"{n_acc/(n*(i+1)):.2f}  q={oracle.n_queries:,}")
    return tracker, x_adv


# --------------------------------------------------- budget re-allocation
def rescue_worst(oracle, x_orig, y, tracker: BestTracker, frac=0.25,
                 geo_steps=600, walk_steps=600, rounds=2, log=print):
    """Spend extra queries only on the images that are still furthest away.

    After the main refinement the distance distribution has a tail: a few
    images sit much further from the boundary than the median, and those are
    exactly the ones where the noise is visible.  Because the oracle is queried
    in batches, running more refinement on a *subset* costs proportionally
    fewer queries, so this is a cheap way to pull the tail in.
    """
    n = x_orig.shape[0]
    k = max(1, int(round(frac * n)))
    for r in range(rounds):
        idx = torch.argsort(tracker.dist, descending=True)[:k]
        sub = BestTracker(x_orig[idx])
        sub.best = tracker.best[idx].clone()
        sub.dist = tracker.dist[idx].clone()
        start = sub.best.clone()
        if geo_steps:
            _, start = geometric_walk(oracle, x_orig[idx], y[idx], start,
                                      steps=geo_steps, tracker=sub)
        if walk_steps:
            _, start = boundary_walk(oracle, x_orig[idx], y[idx],
                                     sub.best.clone(), steps=walk_steps,
                                     tracker=sub)
        improved = sub.dist < tracker.dist[idx]
        sel = idx[improved]
        tracker.best[sel] = sub.best[improved]
        tracker.dist[sel] = sub.dist[improved]
        if log:
            d = tracker.dist * 255
            log(f"    [rescue {r}] worst {k} images: median {d.median():.2f}  "
                f"mean {d.mean():.2f}  max {d.max():.2f}   "
                f"q={oracle.n_queries:,}")
    return tracker


# ------------------------------------------------------- 8-bit quantisation
def quantize_finalize(oracle, x_orig, y, x_adv, tracker: BestTracker,
                      n_dither=8, max_scale=24.0, n_grid=14, bisect=8,
                      generator=None):
    """Turn a continuous adversarial point into one that is *still*
    adversarial after being written to an 8-bit PNG.

    This matters more than it sounds: a converged perturbation is only about
    one pixel level per channel in RMS, i.e. the same order as the
    quantisation step itself, so a point sitting on the continuous boundary
    rounds to the wrong side roughly half the time.  Three families of
    candidates are tried and the closest one the oracle confirms is kept:

      * dithered (stochastic, unbiased) roundings of `x_adv` -- these are free
        in the sense that they do not move away from the clean image;
      * the smallest scaling `c` of the perturbation whose *rounding* is still
        adversarial, found by a geometric sweep followed by bisection. The
        sweep reaches far enough (`max_scale`) that it cannot fail silently
        and leave the image on a distant fallback;
      * dithered roundings at that scale.
    """
    q = oracle.quantize_pixels
    n = x_orig.shape[0]
    dev = x_orig.device
    diff = x_adv - x_orig

    def dithers(base, k):
        for _ in range(k):
            noise = torch.rand(base.shape, device=dev, generator=generator)
            cand = torch.floor(base.clamp(0, 1) * 255.0 + noise).clamp(0, 255) / 255.0
            tracker.update(cand, oracle.is_adversarial(cand, y))

    dithers(x_adv, n_dither)

    # smallest scale whose rounding survives: sweep up, then bisect down.
    # c = 0 is the clean image, which is by definition not adversarial, so it
    # is a valid failing lower bound to bisect against.
    lo = torch.zeros(n, device=dev)
    hi = torch.full((n,), float("inf"), device=dev)
    for i in range(n_grid):
        c = max_scale ** (i / (n_grid - 1))            # 1.0 ... max_scale
        todo = torch.isinf(hi).nonzero(as_tuple=True)[0]
        if todo.numel() == 0:
            break
        cand = q((x_orig[todo] + c * diff[todo]).clamp(0, 1))
        ok = oracle.is_adversarial(cand, y[todo])
        tracker.update(cand, ok, todo)
        hi[todo[ok]] = c
        lo[todo[~ok]] = c

    for _ in range(bisect):
        idx = torch.isfinite(hi).nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            break
        mid = 0.5 * (lo[idx] + hi[idx])
        cand = q((x_orig[idx] + _b(mid) * diff[idx]).clamp(0, 1))
        ok = oracle.is_adversarial(cand, y[idx])
        tracker.update(cand, ok, idx)
        hi[idx] = torch.where(ok, mid, hi[idx])
        lo[idx] = torch.where(ok, lo[idx], mid)

    scale = torch.nan_to_num(hi, posinf=1.0)
    dithers((x_orig + _b(0.9 * scale) * diff).clamp(0, 1), n_dither)
    return tracker


def refine_on_grid(oracle, x_orig, y, tracker: BestTracker, geo_steps=600,
                   walk_steps=600, rounds=1, log=print):
    """Refinement that optimises the quantised objective directly.

    The earlier stages work in continuous pixel space, where the attainable
    optimum can sit *below* the 8-bit grid; what actually gets graded is the
    rounded image. Here the oracle rounds every query onto the grid and the
    tracker measures rounded candidates, so each accepted move is a genuine
    improvement of the submitted .png.
    """
    was = oracle.quantize
    oracle.quantize = True
    tracker.quantize = True
    try:
        for r in range(rounds):
            if geo_steps:
                geometric_walk(oracle, x_orig, y, tracker.best.clone(),
                               steps=geo_steps, tracker=tracker)
            if walk_steps:
                boundary_walk(oracle, x_orig, y, tracker.best.clone(),
                              steps=walk_steps, tracker=tracker)
            if log:
                d = tracker.dist * 255
                log(f"    [grid {r}] median {d.median():.2f}  mean {d.mean():.2f}"
                    f"  max {d.max():.2f}   q={oracle.n_queries:,}")
    finally:
        oracle.quantize = was
    return tracker


# ------------------------------------------------- integer-grid minimisation
def polish_uint8(oracle, x_orig, y, tracker: BestTracker, linf_steps=9,
                 prune_steps=9, greedy_rounds=3, greedy_groups=6,
                 generator=None):
    """Shrink the perturbation directly on the integer pixel grid.

    Every candidate is a valid PNG and every acceptance is oracle-verified,
    so this can only improve the result.
    """
    q = oracle.quantize_pixels
    dev = x_orig.device
    n = x_orig.shape[0]

    def best_delta():
        return (tracker.best - x_orig) * 255.0            # integer-valued

    # 1) smallest L-inf clip radius that keeps the image adversarial ---------
    delta = best_delta()
    hi = delta.abs().flatten(1).max(dim=1).values.clamp(min=1.0)
    lo = torch.zeros(n, device=dev)
    for _ in range(linf_steps):
        mid = torch.floor(0.5 * (lo + hi))
        cand = q(x_orig + torch.clamp(delta, -_b(mid), _b(mid)) / 255.0)
        adv = oracle.is_adversarial(cand, y)
        tracker.update(cand, adv)
        hi = torch.where(adv, mid, hi)
        lo = torch.where(adv, mid + 1.0, lo)
        hi = torch.maximum(hi, lo)

    # 2) prune small-magnitude components -----------------------------------
    delta = best_delta()
    lo = torch.zeros(n, device=dev)
    hi = delta.abs().flatten(1).max(dim=1).values.clamp(min=1.0)
    for _ in range(prune_steps):
        mid = torch.floor(0.5 * (lo + hi))
        pruned = torch.where(delta.abs() <= _b(mid), torch.zeros_like(delta), delta)
        cand = q(x_orig + pruned / 255.0)
        adv = oracle.is_adversarial(cand, y)
        tracker.update(cand, adv)
        lo = torch.where(adv, mid + 1.0, lo)
        hi = torch.where(adv, hi, mid)
        hi = torch.maximum(hi, lo)

    # 3) greedy group-wise shrinking by one quantisation step ---------------
    for _ in range(greedy_rounds):
        delta = best_delta()
        order = torch.rand(delta.shape, device=dev, generator=generator)
        bucket = (order * greedy_groups).long().clamp(0, greedy_groups - 1)
        for g in range(greedy_groups):
            mask = (bucket == g)
            shrunk = torch.where(mask, delta - delta.sign(), delta)
            cand = q(x_orig + shrunk / 255.0)
            adv = oracle.is_adversarial(cand, y)
            took = tracker.update(cand, adv)
            delta = torch.where(_b(took.float()) > 0, shrunk, delta)
    return tracker


# ===================== substitute networks =====================





# --------------------------------------------------------------- architectures
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                       nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        out = out + (x if self.short is None else self.short(x))
        return F.relu(out)


class ResNet(nn.Module):
    """ResNet-18/-10 style CIFAR network with a configurable width."""

    def __init__(self, blocks=(2, 2, 2, 2), width=64, n_cls=10):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(w), nn.ReLU(inplace=True))
        layers, cin = [], w
        for i, nb in enumerate(blocks):
            cout = w * 2 ** i
            for j in range(nb):
                layers.append(BasicBlock(cin, cout, stride=(2 if (j == 0 and i > 0) else 1)))
                cin = cout
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(cin, n_cls)

    def forward(self, x):
        h = self.body(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(h, 1).flatten(1))


class VGGish(nn.Module):
    def __init__(self, n_cls=10, width=64):
        super().__init__()
        cfg = [width, width, "M", 2 * width, 2 * width, "M",
               4 * width, 4 * width, "M", 8 * width, 8 * width, "M"]
        layers, cin = [], 3
        for v in cfg:
            if v == "M":
                layers.append(nn.MaxPool2d(2))
            else:
                layers += [nn.Conv2d(cin, v, 3, padding=1, bias=False),
                           nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                cin = v
        self.features = nn.Sequential(*layers)
        # four 2x2 pools take 32x32 down to 2x2
        self.head = nn.Linear(cin * 4, n_cls)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


def build(name: str) -> nn.Module:
    return {
        "resnet18": lambda: ResNet((2, 2, 2, 2), 64),
        "resnet10": lambda: ResNet((1, 1, 1, 1), 64),
        "resnet18w32": lambda: ResNet((2, 2, 2, 2), 32),
        "vgg": lambda: VGGish(),
        "vggw32": lambda: VGGish(width=32),
    }[name]()


class Normalized(nn.Module):
    """Wraps a surrogate so it takes [0,1] pixel inputs, like the Oracle."""

    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("m", torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer("s", torch.tensor(STD).view(1, 3, 1, 1))

    def forward(self, x01):
        return self.net((x01 - self.m) / self.s)


# -------------------------------------------------------------------- training
def cifar_tensors(root="data/cifar10_hf", train=True, device="cuda"):
    """Public CIFAR-10 split as (uint8 (N,3,32,32) on `device`, labels).

    Uses a local copy if one is present, otherwise downloads the standard
    torchvision copy -- so the same code runs in a fresh notebook.
    """
    split = "train" if train else "test"
    if os.path.exists(os.path.join(root, f"{split}.parquet")) or \
       os.path.exists(os.path.join(root, f"{split}.pt")):
        try:
            return _cifar_load_unavailable(split, root=root, device=device)
        except ImportError:
            pass        # inlined into a notebook: fall through to torchvision
    import numpy as np
    from torchvision import datasets
    ds = datasets.CIFAR10("data/cifar10", train=train, download=True)
    x = torch.from_numpy(np.asarray(ds.data)).permute(0, 3, 1, 2).contiguous()
    y = torch.tensor(ds.targets, dtype=torch.long)
    return x.to(device), y.to(device)


VICTIM_LABEL_CACHE = "data/cifar10_hf/victim_labels.pt"


def victim_labels(oracle=None, model_path="data/p2_data/cnn_model.pt",
                  root="data/cifar10_hf", device="cuda", batch_size=1024,
                  cache=VICTIM_LABEL_CACHE):
    """Label the public CIFAR-10 splits with the *victim's* hard labels.

    This is the substitute-model construction of Papernot et al. (2017): the
    surrogate learns the victim's own decision function from nothing but
    queried labels, so the transfer stage uses no information the black box
    does not hand out.  Cost is a one-off 60,000 queries shared by all 1,000
    attacked images (60 per image), and it is charged to `oracle`'s counter
    when an oracle is passed in.

    Returns (train_labels, test_labels, n_queries_spent).
    """
    if cache and os.path.exists(cache):
        d = torch.load(cache, map_location="cpu", weights_only=True)
        nq = int(d["n_queries"])
        if oracle is not None:      # keep the budget honest on a cache hit
            oracle.n_queries += nq
        return d["train"].to(device), d["test"].to(device), nq

    own = oracle is None
    if own:
        oracle = load_oracle(model_path, device=device, batch_size=batch_size)
    q0 = oracle.n_queries
    out = {}
    for split in ("train", "test"):
        xu, _ = cifar_tensors(root, split == "train", device)
        labs = []
        for i in range(0, xu.shape[0], 5000):
            labs.append(oracle.predict(xu[i:i + 5000].float().div(255.0)).cpu())
        out[split] = torch.cat(labs)
    nq = oracle.n_queries - q0
    out["n_queries"] = torch.tensor(nq)
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        torch.save(out, cache)
    return out["train"].to(device), out["test"].to(device), nq


SURROGATE_SPEC = (("resnet18", 0), ("vgg", 0), ("resnet10", 1), ("vggw32", 2))


def train_or_load(oracle=None, out_dir="data/surrogates", epochs=20, bs=512,
                  lr=0.3, device="cuda", spec=SURROGATE_SPEC, log=print):
    """Return a list of substitute networks, training any that are missing.

    Checkpoints are reused across runs, so the expensive part happens once.
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for name, seed in spec:
        path = os.path.join(out_dir, f"{name}_eps0.0000_s{seed}.pt")
        if not os.path.exists(path):
            log(f"  training substitute {name} (seed {seed}) ...")
            train_one(name, epochs=epochs, bs=bs, lr=lr, device=device,
                      seed=seed, out_dir=out_dir, log=log, oracle=oracle)
        paths.append(path)
    return load_surrogates(paths, device=device), paths


def augment(xb_uint8):
    """Random crop (pad 4) + horizontal flip, done on the GPU."""
    n = xb_uint8.shape[0]
    x = xb_uint8.float().div_(255.0)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (2,))
    x = x[:, :, i[0]:i[0] + 32, i[1]:i[1] + 32]
    flip = torch.rand(n, device=x.device) < 0.5
    x[flip] = torch.flip(x[flip], dims=[3])
    return x


def train_one(name, epochs=30, bs=512, lr=0.2, label_smooth=0.0,
              adv_eps=0.0, device="cuda", seed=0, root="data/cifar10_hf",
              out_dir="data/surrogates", log=print, use_victim_labels=True,
              oracle=None):
    """Train one surrogate.  `adv_eps > 0` enables fast FGSM-RS adversarial
    training (Wong et al., ICLR 2020), giving a *robust* surrogate whose
    gradients transfer to robust victims."""
    torch.manual_seed(seed)
    os.makedirs(out_dir, exist_ok=True)
    xtr, ytr = cifar_tensors(root, True, device)
    xte, yte = cifar_tensors(root, False, device)
    if use_victim_labels:
        vtr, vte, nq = victim_labels(oracle=oracle, root=root, device=device)
        agree = (vtr == ytr).float().mean().item()
        log(f"  [{name}] training on VICTIM labels "
            f"({100*agree:.2f}% agree with ground truth, {nq} queries)")
        ytr, yte = vtr, vte
    net = Normalized(build(name)).to(device).to(memory_format=torch.channels_last)

    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    steps = epochs * (xtr.shape[0] // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps,
                                                pct_start=0.25)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    scaler = torch.amp.GradScaler(device)
    step = 0
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(xtr.shape[0], device=device)
        for i in range(0, xtr.shape[0] - bs + 1, bs):
            idx = perm[i:i + bs]
            xb = augment(xtr[idx]).to(memory_format=torch.channels_last)
            yb = ytr[idx]
            if adv_eps > 0:
                d = (torch.rand_like(xb) * 2 - 1) * adv_eps
                d = (xb + d).clamp(0, 1) - xb
                d.requires_grad_(True)
                with torch.amp.autocast(device):
                    loss = crit(net(xb + d), yb)
                g, = torch.autograd.grad(scaler.scale(loss), d)
                d = (d + 1.25 * adv_eps * g.sign()).clamp(-adv_eps, adv_eps).detach()
                xb = (xb + d).clamp(0, 1)
            with torch.amp.autocast(device):
                loss = crit(net(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if step < steps - 1:
                sched.step()
            step += 1
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            log(f"  [{name}] epoch {ep+1}/{epochs}  loss {loss.item():.3f}  "
                f"test acc {evaluate(net, xte, yte)*100:.2f}%")
    path = os.path.join(out_dir, f"{name}_eps{adv_eps:.4f}_s{seed}.pt")
    torch.save({"name": name, "state": net.state_dict(), "adv_eps": adv_eps}, path)
    log(f"  [{name}] saved -> {path}")
    return net, path


@torch.no_grad()
def evaluate(net, x_uint8, y, bs=512):
    net.eval()
    correct = 0
    for i in range(0, x_uint8.shape[0], bs):
        xb = x_uint8[i:i + bs].float().div(255.0)
        correct += (net(xb).argmax(1) == y[i:i + bs]).sum().item()
    return correct / x_uint8.shape[0]


def load_surrogates(paths, device="cuda"):
    nets = []
    for p in paths:
        ck = torch.load(p, map_location=device, weights_only=False)
        net = Normalized(build(ck["name"])).to(device)
        net.load_state_dict(ck["state"])
        net.eval()
        for q in net.parameters():
            q.requires_grad_(False)
        nets.append(net)
    return nets


# --------------------------------------------------- ensemble gradient / attack
def ensemble_loss_grad(nets, x01, y, targeted_logit_margin=True):
    """Gradient of a margin loss of the surrogate *ensemble* w.r.t. the image.

    Returns a unit-norm direction that, white-box on the surrogates, increases
    the chance of a label flip.  The victim never sees any of this.
    """
    x = x01.clone().requires_grad_(True)
    total = 0.0
    for net in nets:
        logits = net(x)
        if targeted_logit_margin:
            true_logit = logits.gather(1, y.view(-1, 1)).squeeze(1)
            other = logits.scatter(1, y.view(-1, 1), float("-inf")).max(1).values
            total = total + (other - true_logit).mean()
        else:
            total = total - F.cross_entropy(logits, y)
    g, = torch.autograd.grad(total, x)
    return g / (g.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)


def ensemble_pgd_direction(nets, x01, y, eps=16 / 255, steps=30, norm="l2",
                           momentum=1.0):
    """MI-FGSM / PGD on the surrogate ensemble; returns the perturbation
    direction (unit L2 norm) to be line-searched against the victim."""
    x = x01.clone()
    delta = torch.zeros_like(x)
    g_acc = torch.zeros_like(x)
    alpha = 2.5 * eps / steps
    for _ in range(steps):
        xd = (x + delta).clamp(0, 1)
        g = ensemble_loss_grad(nets, xd, y)
        g_acc = momentum * g_acc + g / (g.flatten(1).abs().mean(1).view(-1, 1, 1, 1) + 1e-12)
        if norm == "linf":
            delta = (delta + alpha * g_acc.sign()).clamp(-eps, eps)
        else:
            step = alpha * g_acc / (g_acc.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)
            delta = delta + step
            dn = delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
            delta = delta * torch.clamp(eps / (dn + 1e-12), max=1.0)
        delta = (x + delta).clamp(0, 1) - x
    d = delta
    return d / (d.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)


def ensemble_ddn(nets, x01, y, steps=100, gamma=0.05, init_norm=1.0,
                 quorum=None):
    """Minimum-L2 attack on the surrogate ensemble (Rony et al., CVPR 2019).

    Decoupled direction and norm: follow the loss gradient for the direction
    and shrink/grow the norm depending on whether the current iterate already
    fools the ensemble.  The *direction* of the result is our best white-box
    guess at where the victim's boundary is closest; the victim itself decides
    the radius afterwards.
    """
    n = x01.shape[0]
    quorum = quorum if quorum is not None else max(1, (len(nets) + 1) // 2)
    delta = torch.zeros_like(x01)
    eps = torch.full((n,), init_norm, device=x01.device)
    best = torch.zeros_like(x01)
    best_n = torch.full((n,), float("inf"), device=x01.device)

    for k in range(steps):
        lr = 1.0 * (0.01 ** (k / max(1, steps - 1)))      # cosine-ish decay
        xd = (x01 + delta).clamp(0, 1).requires_grad_(True)
        loss, votes = 0.0, 0
        for net in nets:
            lg = net(xd)
            loss = loss + F.cross_entropy(lg, y, reduction="sum")
            votes = votes + (lg.argmax(1) != y).long()
        g, = torch.autograd.grad(loss, xd)
        g = g / (g.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)

        is_adv = votes >= quorum
        dn = delta.flatten(1).norm(dim=1)
        better = is_adv & (dn < best_n)
        best[better] = delta[better]
        best_n[better] = dn[better]

        delta = delta + lr * g
        eps = torch.where(is_adv, eps * (1 - gamma), eps * (1 + gamma))
        dn = delta.flatten(1).norm(dim=1).clamp(min=1e-12)
        delta = delta * (eps / dn).view(-1, 1, 1, 1)
        delta = ((x01 + delta).clamp(0, 1) - x01).detach()

    # images the ensemble never flipped: fall back to the last iterate
    miss = torch.isinf(best_n)
    best[miss] = delta[miss]
    d = best
    return d / (d.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12), best_n


# ================== transfer-stage directions ==================





# ------------------------------------------------------------ radius search
def radius_search(oracle, x01, y, direction, grid, bsearch=10, tracker=None):
    """Smallest radius r on `grid` (then bisected) with `x + r*dir` adversarial.

    `direction` is unit-L2.  Returns (point, radius); radius is +inf where no
    grid value worked.
    """
    n = x01.shape[0]
    dev = x01.device
    hi = torch.full((n,), float("inf"), device=dev)
    for r in grid:
        todo = torch.isinf(hi).nonzero(as_tuple=True)[0]
        if todo.numel() == 0:
            break
        cand = (x01[todo] + r * direction[todo]).clamp(0, 1)
        ok = oracle.is_adversarial(cand, y[todo])
        if tracker is not None:
            tracker.update(cand, ok, todo)
        hi[todo[ok]] = r

    found = ~torch.isinf(hi)
    lo = torch.zeros(n, device=dev)
    # the previous grid point is a known-failing lower bound
    g = torch.tensor(list(grid), device=dev)
    for i in range(1, len(grid)):
        lo = torch.where(hi == g[i], g[i - 1], lo)

    for _ in range(bsearch):
        idx = found.nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            break
        mid = 0.5 * (lo[idx] + hi[idx])
        cand = (x01[idx] + _b(mid) * direction[idx]).clamp(0, 1)
        ok = oracle.is_adversarial(cand, y[idx])
        if tracker is not None:
            tracker.update(cand, ok, idx)
        hi[idx] = torch.where(ok, mid, hi[idx])
        lo[idx] = torch.where(ok, lo[idx], mid)

    pts = (x01 + _b(torch.nan_to_num(hi, posinf=0.0)) * direction).clamp(0, 1)
    return pts, hi


# ------------------------------------------------------- candidate directions
def candidate_directions(nets, x01, y, chunk=250):
    """A small, diverse set of unit-L2 directions from the surrogate ensemble."""
    specs = [("ens-ddn", None, "ddn", 0.0, 120),
             ("ens-l2", None, "l2", 1.0, 40),
             ("ens-linf", None, "linf", 16 / 255, 40),
             ("ens-grad", None, "grad", 0.0, 0)]
    for i in range(len(nets)):
        specs.append((f"net{i}-ddn", i, "ddn", 0.0, 80))

    out = []
    for name, which, kind, eps, steps in specs:
        sub = nets if which is None else [nets[which]]
        parts = []
        for s in range(0, x01.shape[0], chunk):
            xb, yb = x01[s:s + chunk], y[s:s + chunk]
            if kind == "grad":
                parts.append(ensemble_loss_grad(sub, xb, yb))
            elif kind == "ddn":
                parts.append(ensemble_ddn(sub, xb, yb, steps=steps)[0])
            else:
                parts.append(ensemble_pgd_direction(sub, xb, yb, eps=eps,
                                                      steps=steps, norm=kind))
        out.append((name, torch.cat(parts)))
    return out


def transfer_init(oracle, nets, x01, y, tracker=None, grid=None, bsearch=12,
                  log=print):
    """Line-search every surrogate direction against the victim and keep the
    closest adversarial point found, per image."""
    if grid is None:
        # unit-L2 radii in [0,1] pixel space; 1.0 here == 255 on the 0-255 scale
        grid = (0.05, 0.1, 0.18, 0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.5)
    if tracker is None:
        tracker = BestTracker(x01)

    best = None
    best_d = torch.full((x01.shape[0],), float("inf"), device=x01.device)
    for name, d in candidate_directions(nets, x01, y):
        pts, r = radius_search(oracle, x01, y, d, grid, bsearch, tracker)
        ok = torch.isfinite(r)
        if log:
            rr = r[ok] * 255
            log(f"    [{name:10s}] hit {100*ok.float().mean():5.1f}%  "
                f"radius median {rr.median() if ok.any() else float('nan'):7.1f} "
                f"(0-255)   q={oracle.n_queries:,}")
        upd = ok & (r < best_d)
        if best is None:
            best = pts.clone()
            best_d = torch.where(upd, r, best_d)
            best[~upd] = x01[~upd]
        else:
            best[upd] = pts[upd]
            best_d[upd] = r[upd]
    return best, best_d


# ------------------------------------------- prior-biased probe distribution
def prior_sampler(nets, lam=0.5):
    """Probe directions biased towards the surrogate-ensemble gradient.

    u = normalise( sqrt(lam) * g + sqrt(1-lam) * random ), the mixture used by
    prior-guided random gradient-free methods (P-RGF, Cheng et al. 2019).
    """

    def sample(m, n_samples, shape, device, x_adv, y, k=0, generator=None):
        u = sample_directions(m, n_samples, shape, device, k=k,
                                generator=generator)
        if lam <= 0:
            return u
        g = ensemble_loss_grad(nets, x_adv, y).unsqueeze(1)
        u = (lam ** 0.5) * g + ((1 - lam) ** 0.5) * u
        return u / (u.flatten(2).norm(dim=2).view(m, n_samples, 1, 1, 1) + 1e-12)

    return sample


# ================ robustness of the final images ===============





def input_views(x01, device):
    """The plausible tensors an evaluator ends up feeding the model.

    All of them represent the same 8-bit image; they differ only in memory
    layout and in the last bit of the division by 255.
    """
    u8 = (x01.clamp(0, 1) * 255.0).round().to(torch.uint8)
    nhwc = u8.permute(0, 2, 3, 1).contiguous()          # as an image loader has it
    chw_view = nhwc.permute(0, 3, 1, 2)                 # non-contiguous view
    views = [
        chw_view.float().div(255.0),                    # divided on `device`
        chw_view.contiguous().float().div(255.0),       # contiguous variant
        # divided on the CPU and then moved: not bit-identical to the above
        chw_view.cpu().float().div_(255.0).to(device),
        u8.cpu().float().div_(255.0).to(device),
    ]
    return views


def consensus_adversarial(oracles, x01, y, views=True):
    """Adversarial under every backend and every input representation."""
    device = x01.device
    vs = input_views(x01, device) if views else [x01]
    ok = torch.ones(x01.shape[0], dtype=torch.bool, device=device)
    for o in oracles:
        for v in vs:
            r = o.is_adversarial(v.to(o.device), y.to(o.device)).to(device)
            ok &= r
    return ok


def robustly_adversarial(oracles, x_orig, y, x_cand, shrink=0.97):
    """Accepted only with a radial margin.

    `x_cand` counts as robust when it is adversarial *and* still adversarial
    after pulling the perturbation `1 - shrink` of the way back towards the
    clean image.  A point that survives being shrunk is not sitting on the
    boundary, so it cannot be argued out of it by a last-bit difference.
    """
    quant = lambda t: torch.round(t.clamp(0, 1) * 255.0) / 255.0
    inner = quant(x_orig + shrink * (x_cand - x_orig))
    return (consensus_adversarial(oracles, x_cand, y)
            & consensus_adversarial(oracles, inner, y))


def make_backends(model_path, device, max_batch=2048, cpu=True, loader=None):
    """Deliberately heterogeneous evaluators of the same model file.

    They keep their own query counters, so verification never inflates the
    reported attack budget.  `loader(path, device, batch_size) -> Oracle` lets
    a caller supply its own wrapper.
    """
    loader = loader or load_oracle
    bes = [loader(model_path, device=device, batch_size=max_batch),
           loader(model_path, device=device, batch_size=97),
           loader(model_path, device=device, batch_size=13)]
    if cpu and str(device) != "cpu":
        bes += [loader(model_path, device="cpu", batch_size=512),
                loader(model_path, device="cpu", batch_size=31)]
    return bes


def harden(backends, x_orig, y, x_final, max_passes=40, step=1.05,
           shrink=0.97, log=print):
    """Scale up only the perturbations that are not robustly adversarial.

    Each pass re-checks the *whole* set, so the batch shapes match the ones the
    final check uses, and because a scale only ever grows the loop converges.
    The ladder is fine (5% a pass), so an image that needs help pays a few
    percent of distortion rather than being thrown back to a distant fallback.
    """
    quant = lambda t: torch.round(t.clamp(0, 1) * 255.0) / 255.0
    diff0 = x_final - x_orig
    scale = torch.ones(x_orig.shape[0], device=x_orig.device)
    cur = quant(x_final)
    ok = robustly_adversarial(backends, x_orig, y, cur, shrink)
    n0 = int((~ok).sum())
    for _ in range(max_passes):
        if bool(ok.all()):
            break
        scale = torch.where(ok, scale, scale * step)
        cur = quant(x_orig + scale.view(-1, 1, 1, 1) * diff0)
        ok = robustly_adversarial(backends, x_orig, y, cur, shrink)
    if log and n0:
        log(f"    hardened {n0} image(s) that lacked a margin; "
            f"{int((~ok).sum())} still unresolved")
    return cur, ok



def png_roundtrip(x01):
    """The tensor an evaluator gets after loading our .png files.

    `transforms.ToTensor()` (and any PIL-based loader) produces an HWC uint8
    array that is permuted to CHW, i.e. a *non-contiguous* view.  Feeding a
    differently laid-out tensor to cuDNN can select a different kernel and
    therefore return the other label for a point sitting on the decision
    boundary.  Verifying this exact representation -- not just the contiguous
    tensor we computed -- is what makes the success rate measured here the
    same one measured after the files are read back.
    """
    a = (x01.clamp(0, 1) * 255.0).round().to(torch.uint8)
    nhwc = a.permute(0, 2, 3, 1).contiguous()
    return nhwc.permute(0, 3, 1, 2).float().div(255.0)


def run_attack(oracle, x01, y, cfg):
    """Return (adversarial images on the 8-bit grid, verified mask, query log).

    `x01` are the clean images in [0,1] pixel space, `y` their true labels.
    """
    n = x01.shape[0]
    dev = x01.device
    torch.manual_seed(cfg.get("seed", 0))
    q_log = {}

    def show(tag, dist, extra=""):
        d = dist * 255
        print(f"{tag:26s} L2 median {d.median():8.2f}  mean {d.mean():8.2f}"
              f"  max {d.max():8.2f}   q/img {oracle.n_queries/n:7.0f} {extra}")

    # ---- stage 0: free adversarial seeds (0 queries) -------------------
    # The victim classifies all 1000 provided images correctly, so a clean
    # image carrying a different label is *already* adversarial for this one.
    # The nearest such images are strong starting points that cost nothing.
    seeds = init_seeds(x01, y, n_seeds=cfg["seeds"])

    # The quantised tracker holds the images that will be saved.  Seeding it
    # with a clean 8-bit image of another class puts a 100% success-rate floor
    # under the run: later stages can only replace it with something closer
    # that the oracle has itself confirmed.
    qt = BestTracker(x01)
    s0 = oracle.quantize_pixels(seeds[:, 0])
    qt.update(s0, oracle.is_adversarial(s0, y))

    # ---- stage 1: bisect onto the decision boundary --------------------
    ct = BestTracker(x01)                        # continuous-space refinement
    x_adv, _ = best_initial_boundary_point(oracle, x01, y, seeds,
                                           cfg["bsearch"], ct)
    show("[0-1] seeds -> boundary", ct.dist)
    q_log["init"] = oracle.n_queries

    # ---- stage 1b: substitute-model directions -------------------------
    if cfg.get("transfer"):
        print("[1b]  training substitute networks on victim-queried labels ...")
        nets, _ = train_or_load(oracle=oracle, epochs=cfg["surrogate_epochs"],
                                device=dev)
        pts, rad = transfer_init(oracle, nets, x01, y, tracker=ct)
        better = torch.isfinite(rad) & (rad < l2(x_adv - x01))
        x_adv[better] = pts[better]
        show("[1b]  transfer directions", ct.dist,
             f"({int(better.sum())}/{n} improved)")
    q_log["transfer"] = oracle.n_queries

    # ---- stage 2: HopSkipJump (optional) -------------------------------
    def _log(t, b, k, dist, q):
        if t % 5 == 0 or t == 1:
            print(f"      [hsja] it {t:3d} B={b:4d}  L2 median "
                  f"{(dist*255).median():8.2f}   q/img {q/n:7.0f}")

    if cfg["iters"]:
        hsja(oracle, x01, y, x_adv, iters=cfg["iters"], b0=cfg["b0"],
             b_max=cfg["bmax"], bsearch_steps=cfg["bsearch"], tracker=ct,
             max_batch=cfg["max_batch"], log=_log)
        show("[2]   HopSkipJump", ct.dist)
    q_log["hsja"] = oracle.n_queries

    # ---- stage 3: boundary refinement ----------------------------------
    for r in range(cfg["refine_rounds"]):
        if cfg["geo_steps"]:
            geometric_walk(oracle, x01, y, ct.best.clone(),
                           steps=cfg["geo_steps"], tracker=ct)
        if cfg["walk_steps"]:
            boundary_walk(oracle, x01, y, ct.best.clone(),
                          steps=cfg["walk_steps"], tracker=ct)
        show(f"[3.{r}] boundary refine", ct.dist)
    if cfg.get("rescue_rounds"):
        rescue_worst(oracle, x01, y, ct, frac=cfg["rescue_frac"],
                     geo_steps=cfg["rescue_geo"], walk_steps=cfg["rescue_walk"],
                     rounds=cfg["rescue_rounds"], log=None)
        show("[3r]  rescue the tail", ct.dist)
    q_log["refine"] = oracle.n_queries

    # ---- stage 4: survive 8-bit quantisation ---------------------------
    quantize_finalize(oracle, x01, y, ct.best, qt)
    show("[4]   8-bit quantisation", qt.dist)

    # ---- stage 4b: refine the quantised objective directly -------------
    if cfg.get("grid_rounds"):
        refine_on_grid(oracle, x01, y, qt, geo_steps=cfg["grid_geo"],
                       walk_steps=cfg["grid_walk"], rounds=cfg["grid_rounds"],
                       log=None)
        show("[4b]  grid-space refine", qt.dist)

    # ---- stage 5: integer-grid polish ----------------------------------
    polish_uint8(oracle, x01, y, qt, greedy_rounds=cfg["greedy_rounds"])
    show("[5]   integer-grid polish", qt.dist)
    q_log["finalize"] = oracle.n_queries

    # ---- stage 6: verification that survives a change of machine -------
    # Points on the boundary are separated from the correct label by a
    # numerically tiny margin, so batch size, tensor layout and even the last
    # bit of the division by 255 can flip them back.  Accept only images that
    # keep the wrong label with room to spare: see `robustly_adversarial`.
    x_final = oracle.quantize_pixels(qt.best)

    def _loader(path, device="cpu", batch_size=2048):
        m = oracle.model if str(device) == str(dev) else None
        if m is None:
            m = torch.jit.load(path, map_location=device)
            m.eval()
        return Oracle(m, MEAN, STD, device=device, batch_size=batch_size)

    backends = make_backends(cfg["model_path"], dev, cfg["max_batch"],
                             loader=_loader)
    x_final, verified = harden(backends, x01, y, x_final,
                               shrink=cfg.get("verify_shrink", 0.97))
    if not bool(verified.all()):     # last resort: the free clean-image seed
        bad = (~verified).nonzero(as_tuple=True)[0]
        x_final[bad] = oracle.quantize_pixels(seeds[bad, 0])
        verified = consensus_adversarial(backends, x_final, y)
    print(f"[6]   robustly adversarial on {len(backends)} evaluators x "
          f"{len(input_views(x01[:1], dev))} input representations: "
          f"{int(verified.sum())}/{n}")
    return x_final, verified, q_log


# ---------------------------------------------------------------------
#  Run the attack on the images loaded above
# ---------------------------------------------------------------------
_mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
_std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)

# `X` is normalised; recover the exact 8-bit pixel values in [0, 1].
X01 = (X * _std_t + _mean_t).clamp(0, 1)
X01 = torch.round(X01 * 255.0) / 255.0

oracle = Oracle(model, mean, std, device=device, batch_size=CONFIG["max_batch"])

t0 = time.time()
print(f"clean accuracy: {(oracle.predict(X01) == y_true).float().mean().item()*100:.2f} %")
oracle.reset_counter()

X_adv01, verified, stage_queries = run_attack(oracle, X01, y_true, CONFIG)

print(f"\nattack success rate : {verified.float().mean().item()*100:.2f} %"
      f"  ({int((~verified).sum())} failures)")
d = ((X_adv01 - X01) * 255).flatten(1)
print(f"L2   (0-255 scale)  : mean {d.norm(dim=1).mean():.2f}   "
      f"median {d.norm(dim=1).median():.2f}")
print(f"Linf (0-255 scale)  : mean {d.abs().max(1).values.mean():.2f}   "
      f"median {d.abs().max(1).values.median():.0f}")
_mse = (d ** 2).mean(dim=1).clamp(min=1e-12)
print(f"PSNR (dB)           : mean {(10*torch.log10(255.0**2/_mse)).mean():.2f}")
print(f"model inferences    : {oracle.n_queries:,}  "
      f"({oracle.n_queries/X01.shape[0]:.0f} per image)")
print(f"wall clock          : {time.time()-t0:.0f} s")


## **Save Adversarial Images**
1. You must save the one adversarial image corresponding to each clean image as `.png`.
2. You must follow the naming convention. For example, the adversarial image corresponding to `cat_5.png` must be saved as `cat_5_adv.png`, so that during evaluation they can be compared.
3. Save all the adversarial images in the same folder and name it `adv_images`.
4. Make sure the images are properly denormalized using the same `mean` and `std`. This is essential since it can affect the quality of your adversarial images if not done properly.

In [ ]:
# ---------------------------------------------------------------------
#  Save one adversarial .png per clean image as "<class>_<index>_adv.png"
# ---------------------------------------------------------------------
import os
from PIL import Image

out_dir = "adv_images"
os.makedirs(out_dir, exist_ok=True)

# X_adv01 is already on the 8-bit grid and in [0, 1] pixel space (i.e. the
# denormalisation with `mean`/`std` has been undone), so writing it is lossless.
arr = (X_adv01.clamp(0, 1) * 255.0).round().to(torch.uint8).cpu()
arr = arr.permute(0, 2, 3, 1).numpy()

for a, p in zip(arr, paths):
    name = os.path.splitext(os.path.basename(p))[0]      # e.g. "cat_5"
    Image.fromarray(a, mode="RGB").save(os.path.join(out_dir, f"{name}_adv.png"))

print(f"saved {len(arr)} adversarial images to ./{out_dir}/")

# Re-read the files from disk and re-check them, so the number quoted above is
# the number for the images that are actually being submitted.
reread = torch.stack([
    transforms.ToTensor()(Image.open(os.path.join(out_dir, os.path.splitext(
        os.path.basename(p))[0] + "_adv.png")).convert("RGB")) for p in paths
]).to(device)
_pred = model((reread - _mean_t) / _std_t)
print(f"re-checked from disk: attack success rate "
      f"{(_pred != y_true).float().mean().item()*100:.2f} %")


### **Submission:** Complete this notebook and submit along with adversarial images in adv_images folder.